In [ ]:
import sys, subprocess, os, shutil, glob, json, time, hashlib, zipfile
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap", "sacrebleu"], check=True)
from pycocoevalcap.spice import spice
libs = Path(spice.__file__).resolve().parent / "lib"; libs.mkdir(exist_ok=True)
for name, size in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    have = libs / name
    if not have.is_file() or have.stat().st_size != size:
        c = [p for p in Path("/kaggle/input").rglob(name) if p.stat().st_size == size]
        assert c, f"DUNG: thieu {name}"
        shutil.copy2(c[0], have)
_v = subprocess.run(["bash", "-c", "java -version 2>&1"], capture_output=True, text=True).stdout
if 'version "1.8' not in _v:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-8-jre-headless", shell=True, check=True)
    _j = sorted(glob.glob("/usr/lib/jvm/*8*")); assert _j, "DUNG: khong co Java 8"
    os.environ["JAVA_HOME"] = _j[0]; os.environ["PATH"] = _j[0] + "/bin:" + os.environ["PATH"]
print("Java", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[:1], "· CPU", os.cpu_count(), flush=True)


In [ ]:
W = "/kaggle/working"
D = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sa3_reward_screen.py", recursive=True) if "/notebooks/" not in p]
assert len(D) == 1, D; D = D[0]
for f, h in {'build_branch_data.py': '619e63e123a6dbf60086e65ee94a3912', 'grpo_luoi.py': '92d0a9bc6b8cae8a480aef08f99fb9d8', 'grpo_spice.py': '8489108147c8d1ed80c3dc67c8ad3d3f', 'h4_spice_tuple_7_11.py': 'd5f13d93fb84235f60c4bd2e558a54a5', 'mau_0.jsonl': '738090ee6133a24a0923e3b8cca326d3', 'mau_1.jsonl': '188e4c299722c3660b2d662072630c90', 'sa3_action_verify.py': '672e2b6ff04155a7214e8880509089a2', 'sa3_reward_screen.py': 'f6637048ed5dce6db051b4fabf15833d', 'score_k0_raw.jsonl': '9782ec54b4136edad6ddd82c6c5535a7', 'score_k1_raw.jsonl': '5468429a682d2e17bdd0d9927bd53802', 'score_k2_raw.jsonl': '50616075f853eb60897e0a697b15cf02', 'score_k3_raw.jsonl': 'b01905652395e6f981dc45c76cc84045', 'score_k4_raw.jsonl': '1b39e94258703b7ad9ae15c6d29ad81e', 'score_k5_raw.jsonl': 'babd03716908834a38c6a657b939c8e9', 'score_k6_raw.jsonl': 'e6785730eaba46d55b94ac21e3f2ac41', 'score_k7_raw.jsonl': '1cc6363dd854e97e29afb2bade625279', 'score_k8_raw.jsonl': 'c614191780c01aef07cc4110f69709f0', 'train_v3.jsonl': 'e033ea809a18c0e5e3eb878305396f19'}.items():
    assert hashlib.md5(open(f"{D}/{f}", "rb").read()).hexdigest() == h, f"DUNG: {f} lech md5"
CODE = f"{W}/code"; os.makedirs(CODE, exist_ok=True)
for f in glob.glob(f"{D}/*.py"):
    shutil.copy2(f, CODE)
assert subprocess.run([sys.executable, f"{CODE}/sa3_reward_screen.py", "--selftest"]).returncode == 0, "DUNG: selftest"
OUT = f"{W}/screen"; os.makedirs(OUT, exist_ok=True)
log = f"{W}/screen.log"
cmd = [sys.executable, "-u", f"{CODE}/sa3_reward_screen.py", "--data", D, "--mau", D, "--score", D, "--out", f"{OUT}/sa3_screen.json"]
t0 = time.time()
with open(log, "w") as f:
    p = subprocess.Popen(cmd, cwd=CODE, stdout=f, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    while p.poll() is None:
        time.sleep(120)
        L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
        print(f"{(time.time()-t0)/60:5.1f} phut · {L[-1][:150] if L else 'chua co dong'}", flush=True)
print(open(log, errors="ignore").read()[-6000:], flush=True)
print("ma thoat", p.returncode, flush=True)


In [ ]:
with zipfile.ZipFile(f"{W}/sa3_screen_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{OUT}/*") + [log]:
        z.write(p, os.path.basename(p))
print("zip", os.path.getsize(f"{W}/sa3_screen_out.zip") // 1024, "KB", flush=True)
